# 10. 부스팅 모델 + KOBIS 변수 (TabPFN과 비교 · 블렌딩)

조원들이 부스팅 계열로 좋은 결과를 봐서, **09와 같은 변수(06 최종 + KOBIS, 48개)**로 부스팅 모델을 제대로 비교.

| 단계 | 내용 |
| --- | --- |
| 1 | LightGBM · XGBoost · CatBoost × 타깃 3가지(원본 · sqrt · log) × 설정 2가지(기본 · 작은 데이터용 튜닝) |
| 2 | 가장 좋은 부스팅 모델과 TabPFN(09) 블렌딩: 비율 0~1 비교 + NNLS |
| 3 | 제출 파일: 부스팅 단독 최고 · TabPFN + 부스팅 블렌딩 |

- 검증: 팀 공통 5-Fold (seed 42), 원래 관객수 단위 RMSE
- 참고: 09에서 TabPFN 단독 CV 363,107 / Public **436,672** (현재 최고). 09 가중 평균은 CV는 더 좋았지만 Public은 492,991로 더 나빴음 → **CV 차이가 작으면 리더보드로 확인 필요**

## 0. 설정 · 변수 (09와 동일)

In [1]:
import os
import re
import time
import pickle
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import nnls
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from features import make_features

warnings.filterwarnings('ignore')
train = pd.read_csv('movies_train.csv')
test = pd.read_csv('movies_test.csv')
submission = pd.read_csv('submission.csv')
kobis = pd.read_csv('kobis_movies.csv')
y = train['box_off_num']
hit = y >= 1_000_000
rmse = lambda a, b: mean_squared_error(a, b) ** 0.5
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)
CACHE_DIR = 'cache_10'
os.makedirs(CACHE_DIR, exist_ok=True)

# KOBIS 매칭 (09와 같은 방식)
norm = lambda s: re.sub(r'[^0-9a-zA-Z가-힣]', '', str(s)).lower()
kobis['key'] = kobis['movieNm'].map(norm)
kobis['open_dt'] = pd.to_datetime(kobis['openDt'], errors='coerce')
K_COLS = ['open_scrn', 'open_show', 'max_scrn', 'max_show', 'days_top10', 'best_rank']


def kobis_features(df):
    rows = []
    for title, r in zip(df['title'], pd.to_datetime(df['release_time'])):
        cand = kobis[kobis['key'] == norm(title)]
        row = {f'k_{c}': np.nan for c in K_COLS}
        if len(cand):
            gap = (cand['open_dt'] - r).abs().dt.days.fillna(9999)
            if gap.min() <= 30:
                best = cand.loc[gap.idxmin()]
                row = {f'k_{c}': best[c] for c in K_COLS}
        rows.append(row)
    K = pd.DataFrame(rows)
    K['k_in_top10'] = K['k_max_scrn'].notna().astype(int)
    K['k_best_rank'] = K['k_best_rank'].fillna(11)
    return K.fillna(0).astype(float)


X, X_test = make_features(train, test, {'debut': 1, 'debut_mode': 'nan', 'distributor': 1, 'date': 1, 'rating': 1})
X = pd.concat([X.reset_index(drop=True), kobis_features(train)], axis=1)
X_test = pd.concat([X_test.reset_index(drop=True), kobis_features(test)], axis=1)
print('변수', X.shape[1], '개 (09 TabPFN과 동일)')

변수 48 개 (09 TabPFN과 동일)


## 1. 부스팅 모델 비교
- **기본**: 라이브러리 기본 설정
- **튜닝**: 데이터가 600행이라 과적합을 줄이는 쪽으로 (학습률 ↓ · 트리 수 ↑ · 트리 깊이 ↓ · 행/열 샘플링)

In [2]:
MODELS = {
    ('LGBM', '기본'): lambda: LGBMRegressor(random_state=42, verbose=-1),
    ('LGBM', '튜닝'): lambda: LGBMRegressor(n_estimators=800, learning_rate=0.02, num_leaves=15, min_child_samples=10,
                                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                                           random_state=42, verbose=-1),
    ('XGB', '기본'):  lambda: XGBRegressor(random_state=42),
    ('XGB', '튜닝'):  lambda: XGBRegressor(n_estimators=800, learning_rate=0.02, max_depth=4, min_child_weight=3,
                                          subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, random_state=42),
    ('CAT', '기본'):  lambda: CatBoostRegressor(random_state=42, silent=True),
    ('CAT', '튜닝'):  lambda: CatBoostRegressor(iterations=1500, learning_rate=0.02, depth=5, l2_leaf_reg=3,
                                               random_state=42, silent=True),
}
TARGETS = {
    'raw':  (lambda v: v, lambda p: np.clip(p, 0, None)),
    'sqrt': (np.sqrt,     lambda p: np.square(np.clip(p, 0, None))),
    'log':  (np.log1p,    lambda p: np.expm1(np.clip(p, 0, None))),
}


def run(model_key, target):
    name = f'{model_key[0]}_{model_key[1]}_{target}'
    path = os.path.join(CACHE_DIR, f'{name}.pkl')
    if os.path.exists(path):
        return pickle.load(open(path, 'rb'))
    fwd, inv = TARGETS[target]
    oof, te_pred, fold = np.zeros(len(y)), np.zeros(len(test)), []
    t0 = time.time()
    for tr_idx, va_idx in kf5.split(X):
        m = MODELS[model_key]()
        m.fit(X.iloc[tr_idx], fwd(y.iloc[tr_idx]))
        oof[va_idx] = inv(m.predict(X.iloc[va_idx]))
        te_pred += inv(m.predict(X_test)) / 5          # fold 모델 test 예측 평균
        fold.append(rmse(y.iloc[va_idx], oof[va_idx]))
    res = {'실험': name, '모델': model_key[0], '설정': model_key[1], '타깃': target,
           'CV RMSE': rmse(y, oof), 'fold 표준편차': np.std(fold), '흥행작 RMSE': rmse(y[hit], oof[hit]),
           '흥행작 외 RMSE': rmse(y[~hit], oof[~hit]), '소요(초)': time.time() - t0, 'oof': oof, 'test': te_pred}
    pickle.dump(res, open(path, 'wb'))
    return res


R = {}
for mk in MODELS:
    for t in TARGETS:
        r = run(mk, t)
        R[r['실험']] = r
        print(f"{r['실험']:<16} CV RMSE {r['CV RMSE']:>9,.0f} | 흥행작 {r['흥행작 RMSE']:>9,.0f} | {r['소요(초)']:.0f}초")

LGBM_기본_raw      CV RMSE   639,731 | 흥행작 1,437,308 | 0초
LGBM_기본_sqrt     CV RMSE   544,318 | 흥행작 1,259,043 | 0초


LGBM_기본_log      CV RMSE   679,420 | 흥행작 1,579,775 | 0초


LGBM_튜닝_raw      CV RMSE   525,091 | 흥행작 1,196,589 | 1초


LGBM_튜닝_sqrt     CV RMSE   465,783 | 흥행작 1,076,474 | 1초


LGBM_튜닝_log      CV RMSE   622,569 | 흥행작 1,447,136 | 1초


XGB_기본_raw       CV RMSE   542,897 | 흥행작 1,247,228 | 1초


... (7 outputs left out)

CAT_기본_log       CV RMSE   643,413 | 흥행작 1,494,142 | 9초


CAT_튜닝_raw       CV RMSE   541,809 | 흥행작 1,251,153 | 10초


CAT_튜닝_sqrt      CV RMSE   511,522 | 흥행작 1,186,513 | 10초


CAT_튜닝_log       CV RMSE   696,977 | 흥행작 1,621,140 | 10초


In [3]:
FMT = {'CV RMSE': '{:,.0f}', 'fold 표준편차': '{:,.0f}', '흥행작 RMSE': '{:,.0f}', '흥행작 외 RMSE': '{:,.0f}', '소요(초)': '{:.0f}'}
tbl = pd.DataFrame([{k: v for k, v in r.items() if k not in ('oof', 'test')} for r in R.values()]).sort_values('CV RMSE')
display(tbl.style.format(FMT).highlight_min(subset=['CV RMSE', '흥행작 RMSE', '흥행작 외 RMSE'], color='#c6efce'))
best_boost = R[tbl.iloc[0]['실험']]
print('→ 부스팅 최고:', best_boost['실험'], f"CV {best_boost['CV RMSE']:,.0f}")

,실험,모델,설정,타깃,CV RMSE,fold 표준편차,흥행작 RMSE,흥행작 외 RMSE,소요(초)
9,XGB_튜닝_raw,XGB,튜닝,raw,"446,769","118,954","1,017,759","118,163",6
4,LGBM_튜닝_sqrt,LGBM,튜닝,sqrt,"465,783","105,882","1,076,474","88,704",1
10,XGB_튜닝_sqrt,XGB,튜닝,sqrt,"466,094","149,877","1,076,541","90,506",6
16,CAT_튜닝_sqrt,CAT,튜닝,sqrt,"511,522","159,218","1,186,513","84,924",10
3,LGBM_튜닝_raw,LGBM,튜닝,raw,"525,091","94,044","1,196,589","138,097",1
15,CAT_튜닝_raw,CAT,튜닝,raw,"541,809","164,907","1,251,153","105,906",10
6,XGB_기본_raw,XGB,기본,raw,"542,897","149,061","1,247,228","121,801",1
1,LGBM_기본_sqrt,LGBM,기본,sqrt,"544,318","83,871","1,259,043","100,744",0
7,XGB_기본_sqrt,XGB,기본,sqrt,"548,794","134,402","1,269,636","100,912",1
8,XGB_기본_log,XGB,기본,log,"553,548","131,496","1,277,436","110,344",1


→ 부스팅 최고: XGB_튜닝_raw CV 446,769


## 2. TabPFN(09)과 블렌딩
TabPFN과 부스팅은 구조가 달라서 실수하는 영화도 다를 수 있음 → 섞으면 보완될 여지

In [4]:
pfn = pickle.load(open('cache_09/tabpfn.pkl', 'rb'))
p_oof, p_test = pfn['oof'], pfn['test']
print(f"TabPFN (09) CV {rmse(y, p_oof):,.0f} | 부스팅 최고 CV {best_boost['CV RMSE']:,.0f} | 예측 상관 {np.corrcoef(p_oof, best_boost['oof'])[0,1]:.3f}")

rows = []
for w in np.round(np.arange(0, 1.01, 0.1), 1):
    b = w * p_oof + (1 - w) * best_boost['oof']
    rows.append({'TabPFN 비율': w, 'CV RMSE': rmse(y, b), '흥행작 RMSE': rmse(y[hit], b[hit])})
wdf = pd.DataFrame(rows)
display(wdf.style.format({'TabPFN 비율': '{:.1f}', 'CV RMSE': '{:,.0f}', '흥행작 RMSE': '{:,.0f}'})
        .highlight_min(subset=['CV RMSE'], color='#c6efce'))
best_w = float(wdf.loc[wdf['CV RMSE'].idxmin(), 'TabPFN 비율'])

# 여러 부스팅 모델 + TabPFN NNLS (상위 부스팅 6개까지)
top_boost = list(tbl['실험'].head(6))
O = np.column_stack([p_oof] + [R[n]['oof'] for n in top_boost])
T = np.column_stack([p_test] + [R[n]['test'] for n in top_boost])
w_nnls, _ = nnls(O, y.values.astype(float)); w_nnls = w_nnls / w_nnls.sum()
display(pd.DataFrame({'모델': ['TabPFN'] + top_boost, 'NNLS 가중치': w_nnls}).style.format({'NNLS 가중치': '{:.3f}'}))
print(f'NNLS 블렌딩 CV {rmse(y, O @ w_nnls):,.0f}')

TabPFN (09) CV 363,107 | 부스팅 최고 CV 446,769 | 예측 상관 0.982


,TabPFN 비율,CV RMSE,흥행작 RMSE
0,0.0,"446,769","1,017,759"
1,0.1,"425,809","970,791"
2,0.2,"406,918","928,640"
3,0.3,"390,398","891,989"
4,0.4,"376,559","861,541"
5,0.5,"365,707","837,972"
6,0.6,"358,114","821,874"
7,0.7,"353,989","813,691"
8,0.8,"353,453","813,662"
9,0.9,"356,523","821,786"


,모델,NNLS 가중치
0,TabPFN,0.731
1,XGB_튜닝_raw,0.242
2,LGBM_튜닝_sqrt,0.027
3,XGB_튜닝_sqrt,0.000
4,CAT_튜닝_sqrt,0.000
5,LGBM_튜닝_raw,0.000
6,CAT_튜닝_raw,0.000


NNLS 블렌딩 CV 353,325


## 3. 제출 파일
CV와 리더보드가 어긋난 적이 있어서 후보를 몇 개 만들어 두고 **리더보드로 확인**

In [5]:
def save(pred, fname, desc):
    sub = submission.copy(); sub['box_off_num'] = np.clip(pred, 0, None)
    sub.to_csv(fname, index=False)
    print(f'{fname:<32} {desc}')


save(best_boost['test'], f"10_{best_boost['실험']}.csv", f"부스팅 단독 최고 · CV {best_boost['CV RMSE']:,.0f}")
b = best_w * p_test + (1 - best_w) * best_boost['test']
save(b, f'10_tabpfn{int(best_w*10)}_boost{int(round((1-best_w)*10))}.csv',
     f"TabPFN {best_w:.1f} : 부스팅 {1-best_w:.1f} · CV {wdf['CV RMSE'].min():,.0f}")
save(0.5 * p_test + 0.5 * best_boost['test'], '10_tabpfn5_boost5.csv',
     f"TabPFN 0.5 : 부스팅 0.5 · CV {rmse(y, 0.5*p_oof + 0.5*best_boost['oof']):,.0f}")
tbl.to_csv('10_results.csv', index=False, encoding='utf-8-sig')

10_XGB_튜닝_raw.csv                부스팅 단독 최고 · CV 446,769
10_tabpfn8_boost2.csv            TabPFN 0.8 : 부스팅 0.2 · CV 353,453
10_tabpfn5_boost5.csv            TabPFN 0.5 : 부스팅 0.5 · CV 365,707
